# Equal-Weight S&P 500 Index Fund

## Introduction & Library Imports

The S&P 500 is the world's most popular stock market index. The largest fund that is benchmarked to this index is the SPDR® S&P 500® ETF Trust. It has more than US$250 billion of assets under management.

The goal of this section of the course is to create a Python script that will accept the value of your portfolio and tell you how many shares of each S&P 500 constituent you should purchase to get an equal-weight version of the index fund.

## Library Imports

The first thing we need to do is import the open-source software libraries that we'll be using in this tutorial.

In [1]:
import numpy as np
import pandas as pd
import requests 
import xlsxwriter 
import math 
from io import StringIO

## Importing Our List of Stocks

The next thing we need to do is import the constituents of the S&P 500.

These constituents change over time, so in an ideal world you would connect directly to the index provider (Standard & Poor's) and pull their real-time constituents on a regular basis.

Paying for access to the index provider's API is outside of the scope of this course. 

There's a static version of the S&P 500 constituents available here. [Click this link to download them now](https://drive.google.com/file/d/1ZJSpbY69DVckVZlO9cC6KkgfSufybcHN/view?usp=sharing). Move this file into the `starter-files` folder so it can be accessed by other files in that directory.

Now it's time to import these stocks to our Jupyter Notebook file.

In [2]:
# # stocks = pd.read_csv('sp_500_stocks.csv') 

# # This scrapes the current S&P 500 table from Wikipedia
# sp500_table = pd.read_html('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies')
# stocks = sp500_table[0]

# # Wikipedia calls the column 'Symbol', so we rename it to 'Ticker' to match your code
# stocks.rename(columns={'Symbol': 'Ticker'}, inplace=True)

# url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
# tables = pd.read_html(url)

# df = tables[0]
# stocks = df['Symbol'].tolist()

# stocks = [ticker.replace('.', '-') for ticker in stocks]

url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

response = requests.get(url, headers=headers)

# Wrap response.text in StringIO()
# This prevents the FileNotFoundError by telling pandas: "This is a stream of text"
data_stream = StringIO(response.text)
sp500_table = pd.read_html(data_stream)

stocks = sp500_table[0]
stocks.rename(columns={'Symbol': 'Ticker'}, inplace=True)
stocks['Ticker'] = stocks['Ticker'].str.replace('.', '-', regex=False)

## Acquiring an API Token

Now it's time to import our IEX Cloud API token. This is the data provider that we will be using throughout this course.

API tokens (and other sensitive information) should be stored in a `secrets.py` file that doesn't get pushed to your local Git repository. We'll be using a sandbox API token in this course, which means that the data we'll use is randomly-generated and (more importantly) has no cost associated with it.

[Click here](http://nickmccullum.com/algorithmic-trading-python/secrets.py) to download your `secrets.py` file. Move the file into the same directory as this Jupyter Notebook before proceeding.

In [3]:
import yfinance as yf 

## Making Our First API Call

Now it's time to structure our API calls to IEX cloud. 

We need the following information from the API:

* Market capitalization for each stock
* Price of each stock



In [4]:
symbol = 'AAPL'
api_url = yf.Ticker(symbol) #f'https://sandbox.iexapis.com/stable/stock/{symbol}/quote/?token={IEX_CLOUD_API_TOKEN}' 
data = api_url.info # request.get(api_url).json()


## Parsing Our API Call

The API call that we executed in the last code block contains all of the information required to build our equal-weight S&P 500 strategy. 

With that said, the data isn't in a proper format yet. We need to parse it first.

In [5]:
price = data.get('currentPrice')  # data['latestPrice']
market_cap = data.get('marketCap')  # data['marketCap'] 

## Adding Our Stocks Data to a Pandas DataFrame

The next thing we need to do is add our stock's price and market capitalization to a pandas DataFrame. Think of a DataFrame like the Python version of a spreadsheet. It stores tabular data.

In [6]:
my_columns = [ 'Ticker', 'Stock Price', 'Market Capitalisation', 'Number of Shares to Buy'] 
final_dataframe = pd.DataFrame(columns = my_columns) 

In [7]:
# pd.concat(
#     final_dataframe, 
#     pd.Series(
#         [
#             symbol, 
#             price, 
#             market_cap,
#             'N/A' 
#         ], 
#     index = my_columns 
#     ), 
#     ignore_index = True 
# )
new_row = pd.Series([symbol, price, market_cap, 'N/A'], index = my_columns).to_frame().T
final_dataframe = pd.concat([final_dataframe, new_row], ignore_index = True) 

## Looping Through The Tickers in Our List of Stocks

Using the same logic that we outlined above, we can pull data for all S&P 500 stocks and store their data in the DataFrame using a `for` loop.

In [8]:
final_dataframe = pd.DataFrame(columns = my_columns) 
for stock in stocks['Ticker']: 
    # api_url = f'https://sandbox.iexapis.com/stable/stock/{stock}/quote/?token={IEX_CLOUD_API_TOKEN}' 
    # data = request.get(api_url).json()
    api_url = yf.Ticker(stock) 
    data = api_url.info 
    
    # final_dataframe = pd.concat( [
    #     final_dataframe, 
    #     pd.Series(
    #         [ 
    #             stock, 
    #             data['latestPrice'],
    #             data['marketCap'],
    #             'N/A'
    #         ],
    #         index = my_columns  
    #     ), 
    #     ignore_index = True 
    # )

    new_row = pd.Series([stock, data.get('currentPrice'), data.get('marketCap'), 'N/A'], index = my_columns).to_frame().T 
    final_dataframe = pd.concat([final_dataframe, new_row], ignore_index = True) 

In [9]:
final_dataframe

,Ticker,Stock Price,Market Capitalisation,Number of Shares to Buy
0,MMM,143.29,74735370240,N/A
1,AOS,58.6,8076756992,N/A
2,ABT,84.32,146869682176,N/A
3,ABBV,201.55,356494049280,N/A
4,ACN,180.42,111036809216,N/A
...,...,...,...,...
498,XYL,113.73,27032881152,N/A
499,YUM,151.95,41880641536,N/A
500,ZBRA,226.03,11118800896,N/A
501,ZBH,82.33,15927766016,N/A


## Using Batch API Calls to Improve Performance

Batch API calls are one of the easiest ways to improve the performance of your code.

This is because HTTP requests are typically one of the slowest components of a script.

Also, API providers will often give you discounted rates for using batch API calls since they are easier for the API provider to respond to.

IEX Cloud limits their batch API calls to 100 tickers per request. Still, this reduces the number of API calls we'll make in this section from 500 to 5 - huge improvement! In this section, we'll split our list of stocks into groups of 100 and then make a batch API call for each group.

In [10]:
# def chunks(lst ,n): 
#     """Yield successive n-sized chunks from lst. """ 
#     for i in range(0, len(lst), n): 
#         yield lst[i:i + n]

In [11]:
# symbol_groups = list(chunks(stocks['Ticker'], 100)) 
# symbol_strings = [] 
# for i in range(0, len(symbol_groups)): 
#     symbol_strings.append(','.join(symbol_groups[i]))

# final_dataframe = pd.DataFrame(columns = my_columns)

# for symbol_string in symbol_strings: 
#     batch_api_call_url = f'https://sandbox.iexapis.com/stable/market/batch?symbols={symbol_string}&types=quote&token={IEX_CLOUD_API_TOKEN}' 
#     data = requests.get(batch_api_call_url).json() 
#     for symbol in symbol_strings.spilt(','): 
#         final_dataframe = pd.concat( 
#             final_dataframe, 
#             pd.Series( 
#                 [ 
#                     symbol, 
#                     data[symbol]['quote']['latestPrice'], 
#                     data[symbol]['quote']['marketCap'],
#                     'N/A'
#                 ],
#                 index = my_columns
#             ), 
#             ignore_index = True 
#         ) 

In [12]:
# 1. Initialize the Tickers object with all symbols at once
# stocks['Ticker'] is your list from Wikipedia
tickers_list = stocks['Ticker'].tolist()
tickers_data = yf.Tickers(tickers_list)

rows_list = []

# print("Fetching data... this may take a minute.")

# 2. Loop through the tickers to get 'info'
# We use rows_list to avoid the slow pd.concat inside a loop
for symbol in tickers_list:
    try:
        # Accessing the individual ticker object from our batch
        ticker_info = tickers_data.tickers[symbol].info
        
        price = ticker_info.get('currentPrice') or ticker_info.get('regularMarketPrice')
        market_cap = ticker_info.get('marketCap')
        
        if price:
            rows_list.append({
                'Ticker': symbol,
                'Stock Price': price,
                'Market Capitalisation': market_cap,
                'Number of Shares to Buy': 'N/A'
            })
    except Exception as e:
        # This skips tickers that might have been delisted mid-day or have errors
        continue

# 3. Create the final DataFrame in one go
final_dataframe = pd.DataFrame(rows_list)

# print("Final Dataframe Complete!")
# print(final_dataframe.head())

## Calculating the Number of Shares to Buy

As you can see in the DataFrame above, we stil haven't calculated the number of shares of each stock to buy.

We'll do that next.

In [13]:
portfolio_size = input('Enter the value of your portfolio') 
try: 
    val = float(portfolio_size)
except ValueError: 
    print("That is not a number! \nPlease try again:") 
    portfolio_size = input('Enter the value of your portfolio') 
    val = float(portfolio_size)


Enter the value of your portfolio 10000000


In [21]:
# Force the column to be numeric so it can hold the result of math.floor 
final_dataframe['Number of Shares to Buy'] = pd.to_numeric(final_dataframe['Number of Shares to Buy'], errors='coerce')

position_size = float(portfolio_size) / len(final_dataframe.index) 
for i in range(0, len(final_dataframe.index)): 
    price = final_dataframe.loc[i, 'Stock Price']
    final_dataframe.loc[i, 'Number of Shares to Buy'] = math.floor(position_size / price) 
    

## Formatting Our Excel Output

We will be using the XlsxWriter library for Python to create nicely-formatted Excel files.

XlsxWriter is an excellent package and offers tons of customization. However, the tradeoff for this is that the library can seem very complicated to new users. Accordingly, this section will be fairly long because I want to do a good job of explaining how XlsxWriter works.

### Initializing our XlsxWriter Object

In [22]:
writer = pd.ExcelWriter('recommended_trades.xlsx', engine = 'xlsxwriter') 
final_dataframe.to_excel(writer, sheet_name = 'Recommended Trades', index = False) 

### Creating the Formats We'll Need For Our `.xlsx` File

Formats include colors, fonts, and also symbols like `%` and `$`. We'll need four main formats for our Excel document:
* String format for tickers
* \\$XX.XX format for stock prices
* \\$XX,XXX format for market capitalization
* Integer format for the number of shares to purchase

In [23]:
background_colour = '#0a0a23' 
font_colour = '#ffffff'

string_format = writer.book.add_format(
    { 
        'font_color': font_colour, 
        'bg_color': background_colour, 
        'border': 1 
    } 
)

dollar_format = writer.book.add_format(
    { 
        'num_format': '$0.00', 
        'font_color': font_colour, 
        'bg_color': background_colour, 
        'border': 1 
    } 
)

integer_format = writer.book.add_format(
    { 
        'num_format': '0', 
        'font_color': font_colour, 
        'bg_color': background_colour, 
        'border': 1 
    } 
)

### Applying the Formats to the Columns of Our `.xlsx` File

We can use the `set_column` method applied to the `writer.sheets['Recommended Trades']` object to apply formats to specific columns of our spreadsheets.

Here's an example:

```python
writer.sheets['Recommended Trades'].set_column('B:B', #This tells the method to apply the format to column B
                     18, #This tells the method to apply a column width of 18 pixels
                     string_template #This applies the format 'string_template' to the column
                    )
```

In [24]:
# writer.sheets['Recommended Trades'].set_column('A:A', 18, string_format) 
# writer.sheets['Recommended Trades'].set_column('B:B', 18, string_format) 
# writer.sheets['Recommended Trades'].set_column('C:C', 18, string_format) 
# writer.sheets['Recommended Trades'].set_column('D:D', 18, string_format) 
#writer.close()

This code works, but it violates the software principle of "Don't Repeat Yourself". 

Let's simplify this by putting it in 2 loops:

In [25]:
column_formats = { 
    'A': ['Ticker', string_format], 
    'B': ['Stock Price', dollar_format], 
    'C': ['Market Capitalisation', dollar_format], 
    'D': ['Number of Shares to Buy', integer_format]
} 

for column in column_formats.keys(): 
    writer.sheets['Recommended Trades'].set_column(f'{column}:{column}', 18, column_formats[column][1])
    writer.sheets['Recommended Trades'].write(f'{column}1', column_formats[column][0], string_format)


## Saving Our Excel Output

Saving our Excel file is very easy:

In [26]:
writer.close()